# Exercises for module 5

TF-IDF matrix calculation


## Submission

Create your copy of this Notebook, put in the Google Colab, create share link and put this link on Kampus on Class Exercise #3. Please check if anyone with the link can access your notebook!

# Stuff we already have...
Tokens and BoW

In [1]:
import numpy as np
import pandas as pd

# ----------------------------
# 1) Create a small example dataset (fast to learn)
# ----------------------------
data = [
    ("ham",  "Hey, are we still on for dinner tonight?"),
    ("ham",  "Can you call me when you get a minute?"),
    ("ham",  "I'll be there in 10 minutes."),
    ("ham",  "Happy birthday! Hope you have a great day."),
    ("ham",  "Don't forget the meeting at 9am tomorrow."),
    ("ham",  "Thanks for your help earlier!"),
    ("ham",  "Are you coming to class today?"),
    ("ham",  "Let's study together this weekend."),
    ("ham",  "Please send me the notes from last lecture."),
    ("ham",  "See you soon."),

    ("spam", "WIN a brand new iPhone now! Click here to claim prize."),
    ("spam", "Congratulations! You've won $1000 cash. Reply YES to get it."),
    ("spam", "URGENT: Your account is suspended. Verify now at http://fake.link"),
    ("spam", "Free entry in a weekly competition! Text WIN to 80085."),
    ("spam", "You have been selected for a FREE gift card. Claim today."),
    ("spam", "Earn money fast from home!!! Limited offer, sign up now."),
    ("spam", "Cheap meds available online. Buy now and save big."),
    ("spam", "Winner! Call this number to collect your reward immediately."),
    ("spam", "Exclusive deal: 90% discount if you act now."),
    ("spam", "LAST CHANCE: Click to unlock your prize."),
]


# demo data
data = [
    ("ham",  "Hey, are you still on for dinner tonight?"),
    ("ham",  "Can you call me when you get a minute?"),
    ("spam", "Hey, are you ready to unlock your prize."),
    ("spam", "You are selected to get FREE prize.")
]

df = pd.DataFrame(data, columns=["label", "text"])
df["y"] = (df["label"] == "spam").astype(int)  # spam=1, ham=0

# ----------------------------
# 2) Text preprocessing + Bag-of-Words vectorizer (Pandas + NumPy)
# ----------------------------
def simple_tokenize(text: str):
    """
    Very small tokenizer:
    - lowercase
    - keep letters/digits
    - split on whitespace
    """
    text = text.lower()
    cleaned = []
    for ch in text:
        # we store alphanumeric chars and spaces
        if ch.isalnum() or ch.isspace():
            cleaned.append(ch)
        else:
            cleaned.append(" ")
    return [tok for tok in "".join(cleaned).split() if tok]


def build_vocab(texts, min_count=1):
    """
    Build vocabulary dict token->index using Pandas Series operations.
    """
    s = pd.Series(texts)
    # this is that cool python feature which allows passing
    # function as parameter to apply function, which then
    # well, applies, this function to every element in the
    # dataframe column or Pandas series (like here)
    tokens = s.apply(simple_tokenize)

    # Flatten list-of-lists into one list
    all_tokens = tokens.explode().dropna()
    counts = all_tokens.value_counts()

    vocab_tokens = counts[counts >= min_count].index.tolist()
    vocab = {tok: i for i, tok in enumerate(vocab_tokens)}
    return vocab, counts

vocab, token_counts = build_vocab(df["text"], min_count=1)


# Build Bag of Words array

def vectorize_bow(texts, vocab):
    """
    Convert list of texts to bag-of-words matrix X: shape (n_samples, vocab_size)
    """
    n = len(texts)
    V = len(vocab)
    X = np.zeros((n, V), dtype=np.float64)

    for i, text in enumerate(texts):
        for tok in simple_tokenize(text):
            j = vocab.get(tok)
            if j is not None:
                X[i, j] += 1.0
    return X


bow = vectorize_bow(df["text"], vocab)

texts = df["text"]
print(texts)
print(bow)


0    Hey, are you still on for dinner tonight?
1       Can you call me when you get a minute?
2     Hey, are you ready to unlock your prize.
3          You are selected to get FREE prize.
Name: text, dtype: object
[[1. 1. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [2. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1. 0. 1. 1. 0. 1. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0.]
 [1. 1. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1.]]


# Exercise: From Bag-of-Words to TF-IDF

## Goal

In this exercise, we will:

1. Understand what **TF-IDF** is and why it is used in NLP
2. Learn how to compute TF-IDF step by step
3. Implement TF-IDF using your existing Bag-of-Words pipeline


## What is TF-IDF?

So far, we have built a **Bag-of-Words (BoW)** representation:

* Each document is represented by counts of words
* Example: `"dog cat dog"` → `{dog: 2, cat: 1}`

**Problem**

* Common words (e.g., *"the"*, *"is"*) dominate counts
* Rare but meaningful words are undervalued


### TF-IDF intuition

TF-IDF stands for:

* **TF (Term Frequency)** → how often a word appears in a document
* **IDF (Inverse Document Frequency)** → how rare the word is across all documents

**Idea:**

* Words are **important** if:

  * they appear often in a document
  * but not in many other documents


## 2. TF-IDF Formula

### Term Frequency (TF)

For word $t$ in document $d$:

$$
TF(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total number of words in } d}
$$


### Inverse Document Frequency (IDF)

Let:

* $N$ = total number of documents
* $df(t)$ = number of documents containing word $t$

$$
IDF(t) = \log\left(\frac{N}{df(t)}\right)
$$

Often we use smoothing:

$$
IDF(t) = \log\left(\frac{N + 1}{df(t) + 1}\right) + 1
$$


### TF-IDF

$$
TFIDF(t, d) = TF(t, d) \cdot IDF(t)
$$


## 3. Algorithm (Step-by-Step)

Given:

* `texts` (list of documents)
* `vocab` (token → index)


### Step 1: Tokenize all documents

Reuse:

```python
tokens = [simple_tokenize(text) for text in texts]
```


### Step 2: Compute document frequency (DF)

For each token:

* Count in how many documents it appears

```python
df_counts[token] = number of documents containing token
```


### Step 3: Compute IDF

For each token:

$$
IDF(t) = \log\left(\frac{N + 1}{df(t) + 1}\right) + 1
$$


### Step 4: Compute TF for each document

For each document:

* Count tokens
* Normalize by document length


### Step 5: Compute TF-IDF vectors

For each document and token:

$$
TFIDF(t, d) = TF(t, d) \cdot IDF(t)
$$

Store result as a NumPy vector of size `len(vocab)`


## 4. Your Tasks

We will concentrate on the simpler part and combine it with the full solution

### Task 1: Compute Document Frequencies

Implement:

```python
def compute_df(texts, vocab):
    # returns dict: token -> document frequency
    pass
```

### Task 2: Compute IDF

```python
def compute_idf(df_counts, N):
    # returns dict: token -> idf value
    pass
```


## 5. Hints

* Use `set(tokens)` when computing DF (to avoid double counting)
* Use `np.zeros(len(vocab))` for document vectors
* Use `np.log()` for logarithm
* Be careful with division by zero (use smoothing!)


## Questions

1. Why does TF-IDF reduce the importance of common words?
2. What happens if a word appears in every document?
3. How would TF-IDF behave for very short documents?

## Bonus (Optional)

* Normalize TF-IDF vectors using L2 norm:

$$
|v| = \sqrt{\sum_i v_i^2}
$$



### Task 1

Compute first building block - document frequencies


In [2]:


def compute_df(texts, vocab):
    """
    Compute document frequency for each token in vocab.
    Document frequency counts in how many different documents a token appears

    Parameters
    ----------
    texts : iterable of str
        Collection of documents
    vocab : dict
        token -> index

    Returns
    -------
    df_counts : dict
        token -> number of documents containing this token
    """
    # let's set zero for all counts for all tokens
    
    df_counts = {token: 0 for token in vocab}
   
    for message in texts:
        tokens = simple_tokenize(message)
        unique_tokens = set(tokens)

        for word in unique_tokens:
            if word in vocab:
                df_counts[word] += 1

    return df_counts

    
    
    # for every text in texts get tokens (use simple_tokenize)
    # then get only unique tokens - recall what structure keeps only unique values
    # for each uniqu token and each token that is present in vocab increas its count by 1
    # return df_counts



compute_df(texts, vocab)

{'you': 4,
 'are': 3,
 'hey': 2,
 'prize': 2,
 'to': 2,
 'get': 2,
 'for': 1,
 'on': 1,
 'still': 1,
 'dinner': 1,
 'call': 1,
 'me': 1,
 'can': 1,
 'tonight': 1,
 'a': 1,
 'when': 1,
 'ready': 1,
 'minute': 1,
 'unlock': 1,
 'your': 1,
 'selected': 1,
 'free': 1}

### Task 2: Compute IDF

Use the smoothed IDF formula:

$$
IDF(t) = \log\left(\frac{N + 1}{df(t) + 1}\right) + 1
$$

where:

* $N$ = number of documents
* $df(t)$ = number of documents containing token $t$

In [4]:
def compute_idf(df_counts, N):
    """
    Compute inverse document frequency for each token.

    Parameters
    ----------
    df_counts : dict
        token -> document frequency
    N : int
        total number of documents

    Returns
    -------
    idf : dict
        token -> idf value
    """
    # we will store here token -> IDF
    idf = {}

    # for each token and count that is in df_counts (use df_counts.items())
    # use the formula for IDF, idf[token] =...
    # return idf
    for tok , count in df_counts.items():
        idf[tok] = np.log((N + 1)/(count+1))+1

    return idf



compute_idf(compute_df(texts, vocab), len(texts))

{'you': np.float64(1.0),
 'are': np.float64(1.2231435513142097),
 'hey': np.float64(1.5108256237659907),
 'prize': np.float64(1.5108256237659907),
 'to': np.float64(1.5108256237659907),
 'get': np.float64(1.5108256237659907),
 'for': np.float64(1.916290731874155),
 'on': np.float64(1.916290731874155),
 'still': np.float64(1.916290731874155),
 'dinner': np.float64(1.916290731874155),
 'call': np.float64(1.916290731874155),
 'me': np.float64(1.916290731874155),
 'can': np.float64(1.916290731874155),
 'tonight': np.float64(1.916290731874155),
 'a': np.float64(1.916290731874155),
 'when': np.float64(1.916290731874155),
 'ready': np.float64(1.916290731874155),
 'minute': np.float64(1.916290731874155),
 'unlock': np.float64(1.916290731874155),
 'your': np.float64(1.916290731874155),
 'selected': np.float64(1.916290731874155),
 'free': np.float64(1.916290731874155)}

## Now let's calculate TF and final TF-IDF

This is more work, so let's jump to the solutions right away

### Compute Term Frequency (TF) matrix.

In [5]:
def compute_tf(texts, vocab):
    """
    Compute Term Frequency (TF) matrix.

    Parameters
    ----------
    texts : iterable of str
        Collection of documents
    vocab : dict
        token -> index

    Returns
    -------
    tf_matrix : np.ndarray
        TF matrix of shape (num_docs, vocab_size)
    """
    num_docs = len(texts)
    vocab_size = len(vocab)

    tf_matrix = np.zeros((num_docs, vocab_size), dtype=float)

    for doc_idx, text in enumerate(texts):
        tokens = simple_tokenize(text)

        if len(tokens) == 0:
            continue

        # Count tokens
        token_counts = {}
        for token in tokens:
            if token in vocab:
                token_counts[token] = token_counts.get(token, 0) + 1

        doc_length = len(tokens)

        # Fill TF values
        for token, count in token_counts.items():
            tf_matrix[doc_idx, vocab[token]] = count / doc_length

    return tf_matrix

### Compute TF-IDF matrix

Now we reuse TF and just multiply by IDF.

In [6]:
def compute_tfidf(texts, vocab, idf):
    """
    Compute TF-IDF matrix using precomputed IDF.

    Parameters
    ----------
    texts : iterable of str
    vocab : dict
        token -> index
    idf : dict
        token -> idf value

    Returns
    -------
    X : np.ndarray
        TF-IDF matrix
    """
    # Step 1: compute TF
    tf_matrix = compute_tf(texts, vocab)

    # Step 2: convert idf dict -> aligned vector
    vocab_size = len(vocab)
    idf_vector = np.zeros(vocab_size, dtype=float)

    for token, idx in vocab.items():
        idf_vector[idx] = idf[token]

    # Step 3: apply TF-IDF (broadcasting)
    tfidf_matrix = tf_matrix * idf_vector

    return tfidf_matrix

In [7]:
texts = df["text"].tolist()


idf = compute_idf(compute_df(texts, vocab), len(texts))
tf_matrix = compute_tf(texts, vocab)
tfidf_matrix = compute_tfidf(texts, vocab, idf)

print("TF shape:", tf_matrix.shape)
print("TF-IDF shape:", tfidf_matrix.shape)

print(tfidf_matrix)

TF shape: (4, 22)
TF-IDF shape: (4, 22)
[[0.125      0.15289294 0.1888532  0.         0.         0.
  0.23953634 0.23953634 0.23953634 0.23953634 0.         0.
  0.         0.23953634 0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.22222222 0.         0.         0.         0.         0.16786951
  0.         0.         0.         0.         0.21292119 0.21292119
  0.21292119 0.         0.21292119 0.21292119 0.         0.21292119
  0.         0.         0.         0.        ]
 [0.125      0.15289294 0.1888532  0.1888532  0.1888532  0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.23953634 0.
  0.23953634 0.23953634 0.         0.        ]
 [0.14285714 0.17473479 0.         0.21583223 0.21583223 0.21583223
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.27375582 0.27375582]]


# Some considerations...


## Why TF-IDF helps logistic regression

Logistic regression is a **linear model**, which means:

* It assigns a weight to each feature (word)
* Prediction is based on a weighted sum of features

With plain Bag-of-Words (counts):

* Frequent but uninformative words (e.g., *“the”*, *“and”*) get large values
* This can **dominate the model** and reduce performance

👉 TF-IDF fixes this by:

* **Downweighting common words**
* **Upweighting rare, informative words**


## Intuition: What changes?

### Without TF-IDF (BoW counts)

| word  | count |
| ----- | ----- |
| the   | 10    |
| great | 1     |

Model may think *“the”* is important (just because it’s frequent)


### With TF-IDF

| word  | tf-idf |
| ----- | ------ |
| the   | low    |
| great | high   |

Model now focuses on meaningful signals


## How it affects logistic regression mathematically

Logistic regression computes:

$$
z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b
$$

Then applies sigmoid:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

👉 If you replace:

* $x_i$ = **raw counts** → BoW
* $x_i$ = **TF-IDF values** → weighted importance

You are directly changing the **input feature space**, which strongly affects learning.


## Practical benefits

Using TF-IDF with logistic regression:

* Improves classification accuracy (often significantly)
* Reduces noise from stopwords
* Makes features more comparable across documents
* Works especially well for:

  * sentiment analysis
  * topic classification
  * spam detection



## When TF-IDF might not help

* Very small datasets (can overfit)
* When using models that already learn representations (e.g., neural networks, transformers)
* When raw frequency itself carries meaning (rare cases)

# TF-IDF Solution (Vectorized with Pandas/NumPy)

Instead of:

* looping over documents manually

we:

* use **Pandas `Series`, `explode`, `groupby`, `value_counts`**
* build matrices using **NumPy arrays**


In [8]:
texts = df["text"]
tokens_series = texts.apply(simple_tokenize)

## Task 1: Compute Document Frequency (vectorized)

def compute_df_fast(tokens_series):
    """
    Compute document frequency using Pandas operations.
    """
    # Convert each document to unique tokens
    unique_tokens = tokens_series.apply(set)

    # Flatten
    exploded = unique_tokens.explode()

    # Count occurrences across documents
    df_counts = exploded.value_counts()

    return df_counts.to_dict()


## Task 2: Compute IDF (vectorized)

def compute_idf_fast(df_counts, vocab, N):
    """
    Compute IDF aligned with vocab order.
    """
    idf = np.zeros(len(vocab))

    for token, idx in vocab.items():
        df_t = df_counts.get(token, 0)
        idf[idx] = np.log((N + 1) / (df_t + 1)) + 1

    return idf

## Task 3: Compute TF matrix (vectorized core)

### Step A: Build count matrix using Pandas

def compute_tf_matrix(tokens_series, vocab):
    """
    Build normalized TF matrix using Pandas.
    """
    # Explode tokens
    df_tokens = tokens_series.explode().reset_index()
    df_tokens.columns = ["doc_id", "token"]

    # Keep only tokens in vocab
    df_tokens = df_tokens[df_tokens["token"].isin(vocab)]

    # Map token -> column index
    df_tokens["token_id"] = df_tokens["token"].map(vocab)

    # Count occurrences
    counts = (
        df_tokens
        .groupby(["doc_id", "token_id"])
        .size()
        .unstack(fill_value=0)
    )

    # Ensure all columns exist
    counts = counts.reindex(columns=range(len(vocab)), fill_value=0)

    # Normalize to TF
    doc_lengths = counts.sum(axis=1).values.reshape(-1, 1)
    doc_lengths[doc_lengths == 0] = 1

    tf = counts.values / doc_lengths

    return tf


## Final: Compute TF-IDF (fully vectorized)

def compute_tfidf_fast(texts, vocab):
    tokens_series = texts.apply(simple_tokenize)

    # DF
    df_counts = compute_df_fast(tokens_series)

    # IDF
    idf = compute_idf_fast(df_counts, vocab, N=len(texts))

    # TF
    tf = compute_tf_matrix(tokens_series, vocab)

    # TF-IDF (broadcasting)
    tfidf = tf * idf

    return tfidf


## Usage


tfidf_matrix = compute_tfidf_fast(df["text"], vocab)

print("Shape:", tfidf_matrix.shape)
print(tfidf_matrix)

Shape: (4, 22)
[[0.125      0.15289294 0.1888532  0.         0.         0.
  0.23953634 0.23953634 0.23953634 0.23953634 0.         0.
  0.         0.23953634 0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.22222222 0.         0.         0.         0.         0.16786951
  0.         0.         0.         0.         0.21292119 0.21292119
  0.21292119 0.         0.21292119 0.21292119 0.         0.21292119
  0.         0.         0.         0.        ]
 [0.125      0.15289294 0.1888532  0.1888532  0.1888532  0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.23953634 0.
  0.23953634 0.23953634 0.         0.        ]
 [0.14285714 0.17473479 0.         0.21583223 0.21583223 0.21583223
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.27375582 0.27375582]]


## What is `pandas.Series.explode()`?

`explode()` is a very useful function in pandas that:

👉 **takes a column (or Series) containing lists and “unpacks” them into multiple rows**



### Intuition

You start with something like this:

| doc_id | tokens             |
| ------ | ------------------ |
| 0      | ["this", "is"]     |
| 1      | ["another", "doc"] |

After `explode()`:

| doc_id | tokens  |
| ------ | ------- |
| 0      | this    |
| 0      | is      |
| 1      | another |
| 1      | doc     |


### Example

```python
import pandas as pd

s = pd.Series([
    ["this", "is"],
    ["another", "document"]
])

print(s)
```

Output:

```text
0         [this, is]
1    [another, document]
dtype: object
```


In our case

```python
tokens = texts.apply(simple_tokenize)
```

You get:

```text
0    ["this", "is", "text"]
1    ["another", "text"]
```

After:

```python
tokens.explode()
```

We get a **flat list of tokens**:

```text
0    this
0    is
0    text
1    another
1    text
```

Now we can easily:

```python
tokens.explode().value_counts()
```

Count all words across corpus